# Uniform ORN / PN / LN Lateralization — 7 Antennal Lobes across 4 Connectomes

Three **contra / ipsi synapse-count ratios** per glomerulus, computed identically across
**4 animals** (Male-CNS, FAFB/FlyWire, BANC, hemibrain) split into **7 antennal-lobe series**:
`MCNS-L/R`, `FAFB-L/R`, `BANC-L/R`, `hemibrain-R`.

The **unit of analysis is one complete antennal lobe.** For each AL, every ORN→target synapse\n*in that AL* is labelled **ipsi** (ORN soma on the same side) or **contra** (other side):\n\n| Metric | Definition (within one AL) |\n|---|---|\n| **R_contra** | ORN axon: contra-antenna ORN output / ipsi-antenna ORN output |\n| **P_PN** | ORN→**uniglomerular** ALPN synapses from contra vs ipsi antenna |\n| **P_LN** | ORN→**GABAergic** ALLN synapses from contra vs ipsi antenna |\n\n> **⚠ PN/LN filters (v2):** P_PN is now restricted to **uniglomerular** PNs only;\n> P_LN is restricted to **GABAergic** LNs only. Filtering details per dataset:\n>\n> * **MCNS** — PN: flywireType starts with a known glomerulus name (excludes `M_*`, `+`,\n>   `CB*`). LN: `predictedNt == 'gaba'`.\n> * **FAFB** — PN: `sub_class == 'uniglomerular'`. LN: cross-references\n>   `neurons.csv.gz` for `nt_type == 'GABA'`.\n> * **BANC** — PN: `Sub Class == 'uniglomerular_projection_neuron'`. LN:\n>   `Predicted NT type == 'GABA'`.\n> * **Hemibrain** — PN: type does not start with `M_`. LN: cross-references\n>   `hemibrain_with_nt` for max-NT = GABA (dominant predicted NT).\n\nBecause ipsi/contra is the **AL-of-the-synapse vs the ORN's soma side**, bilateral PNs\n(`V_ilPN` …) are scored correctly, and **hemibrain works from its single complete right AL**:\nits contralateral (`_L`) ORN arbors are reconstructed there, giving real contra input; R_contra\nuses mirror symmetry (contra-ORN output into the right AL ≈ right-ORN output into the absent\nleft AL).\n\n**Caveats:** L and R of one brain are mirror **pseudo-replicates** (4 animals, 7 ALs — not 7\nindependent samples); hemibrain R_contra assumes mirror symmetry; hemibrain PN/LN identity is\nby type-string (no `class` field). The per-ORN plot (§7) excludes hemibrain.\n\nData: FAFB/BANC use local per-edge `neuropil`; MCNS & hemibrain use cached neuPrint fetches\n(`data/Male_CNS/orn_to_alpnln_AL_roi_adj.feather`, `data/Hemibrain/orn_lat_syn_upn_gaba.feather`).\n**Plots** use the raw `contra/ipsi` ratio: **0 = fully ipsi (origin), 1 = balanced, >1 = contra**;\nextrema (incl. `ipsi=0`→∞) are clipped to a per-plot cap and marked. CSV keeps `ratio` + `log2_ratio`.

In [ ]:
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns

sns.set_context('notebook')
sns.set_style('whitegrid')
# setup dark background for all subsequent plots (overridden by plt.style.use('default') if needed):
plt.style.use('dark_background')
DATA = '../data'
EPS = 1e-9

# --- uniform helpers -------------------------------------------------------
_SIDE = {'L': 'left', 'R': 'right', 'left': 'left', 'right': 'right'}
def norm_side(s):
    '''Map L/R/left/right -> left/right; everything else (M/midline/unknown/nan) -> None.'''
    return _SIDE.get(str(s).strip(), None)

def glom(t):
    '''ORN type name -> glomerulus label, e.g. "ORN_DA1" -> "DA1". Non-ORN -> None.'''
    t = str(t)
    return t[4:] if t.startswith('ORN_') else None

def is_orn(t):
    return str(t).startswith('ORN_')

def log2r(contra, ipsi, pc=1.0):
    '''Symmetric log2 contra/ipsi ratio with a pseudocount (robust to zeros).'''
    return np.log2((np.asarray(contra, float) + pc) / (np.asarray(ipsi, float) + pc))

def cratio(contra, ipsi):
    '''Raw contra/ipsi ratio: 0 = fully ipsilateral, 1 = balanced, >1 = contra-dominant.
    ipsi==0 with contra>0 -> +inf (a "fully contra" extremum); 0/0 -> NaN.'''
    contra = np.asarray(contra, float); ipsi = np.asarray(ipsi, float)
    with np.errstate(divide='ignore', invalid='ignore'):
        r = contra / ipsi
    r[(ipsi == 0) & (contra == 0)] = np.nan
    return r

def cap_extreme(vals, cap):
    '''For plotting on a raw-ratio axis: clip values to `cap` and flag which were
    extreme (non-finite or > cap) so they can be drawn/marked separately.'''
    v = np.asarray(vals, float)
    extreme = ~np.isfinite(v) | (v > cap)
    return np.where(extreme, cap, v), extreme

## 1. Per-dataset loaders
Each returns `(metric1_df, edges23_df)` with identical schemas.

In [ ]:
"""Each loader returns a uniform long SYNAPSE table with one row per
(glomerulus, orn_side, syn_AL, orn_id) aggregate, columns:
    animal, glomerulus, syn_AL ('left'/'right'), orn_side, orn_id, weight, kind
where kind = 'out' (ORN total output in that AL -> R_contra), 'PN', or 'LN'.
The unit of analysis is one complete antennal lobe; ipsi/contra is decided later
as (orn_side == syn_AL).

**PN/LN filtering:** P_PN is limited to *uniglomerular* PNs only; P_LN is limited
to *GABAergic* LNs only. See each loader's docstring for dataset-specific rules."""

def load_mcns():
    '''Male-CNS (neuPrint). R_contra from roiInfo AL(L)/AL(R) 'downstream'; P_PN/P_LN from the
    cached per-ROI neuPrint adjacency. neurons.pkl is the user's own cached fetch (trusted).

    PN filter (uniglomerular): ALPN flywireType must start with a known glomerulus name
    (e.g. DA1_, DL2d_), excluding M_ (multiglomerular), + (polyglomerular), and CB (unknown).
    LN filter (GABAergic): ALLN predictedNt == 'gaba'.'''
    with open(f'{DATA}/Male_CNS/neurons.pkl', 'rb') as f:
        meta = pickle.load(f)[0].copy()
    side = meta['rootSide'].map(norm_side).fillna(meta['somaSide'].map(norm_side))
    meta['side'] = side
    cls = meta['class'].astype(str); typ = meta['flywireType'].astype(str)

    # -- uniglomerular PN filter (flywireType heuristic) ---------------------
    # Heuristic: uniglomerular types start with a known glomerulus name.
    _GLOM_PREFIXES = {'DA1_','DA2_','DA3_','DA4l_','DA4m_',
                      'DC1_','DC2_','DC3_','DC4_',
                      'DL1_','DL2d_','DL2v_','DL3_','DL4_','DL5_',
                      'DM1_','DM2_','DM3_','DM4_','DM5_','DM6_',
                      'DP1l_','DP1m_','D_',
                      'V_','VA1d_','VA1v_','VA2_','VA3_','VA4_','VA5_','VA6_','VA7l_','VA7m_',
                      'VC1_','VC2_','VC3_','VC4_','VC5_',
                      'VL1_','VL2a_','VL2p_',
                      'VM1_','VM2_','VM3_','VM4_','VM5d_','VM5v_','VM6_','VM7d_','VM7v_',
                      'VP1d_','VP1l_','VP1m_','VP2_','VP3_','VP4_','VP5_','MZ_'}
    is_upn = cls.eq('ALPN') & typ.str.startswith(tuple(sorted(_GLOM_PREFIXES)))
    # -- GABAergic LN filter --------------------------------------------------
    is_gaba_ln = cls.eq('ALLN') & (meta['predictedNt'].astype(str) == 'gaba')

    meta['role'] = np.select([typ.map(is_orn), is_upn, is_gaba_ln],
                             ['ORN', 'ALPN', 'ALLN'], default=None)
    meta['glomerulus'] = np.where(meta['role'] == 'ORN', typ.map(glom), None)

    def alfield(roi, key, field):
        d = roi.get(key, {}) if isinstance(roi, dict) else {}
        return d.get(field, 0) or 0

    orns = meta[(meta['role'] == 'ORN') & meta['side'].notna() & meta['glomerulus'].notna()].copy()
    orns['alL'] = orns['roiInfo'].apply(lambda r: alfield(r, 'AL(L)', 'downstream'))
    orns['alR'] = orns['roiInfo'].apply(lambda r: alfield(r, 'AL(R)', 'downstream'))
    out = pd.concat([
        pd.DataFrame({'glomerulus': orns['glomerulus'].values, 'orn_side': orns['side'].values,
                      'syn_AL': 'left', 'orn_id': orns['bodyId'].values, 'weight': orns['alL'].values, 'kind': 'out'}),
        pd.DataFrame({'glomerulus': orns['glomerulus'].values, 'orn_side': orns['side'].values,
                      'syn_AL': 'right', 'orn_id': orns['bodyId'].values, 'weight': orns['alR'].values, 'kind': 'out'}),
    ], ignore_index=True)

    # P_PN/P_LN: ORN->ALPN/ALLN resolved by AL(L)/AL(R) (cached neuPrint fetch)
    postn = meta[meta['role'].isin(['ALPN', 'ALLN']) & meta['side'].notna()]
    cache = f'{DATA}/Male_CNS/orn_to_alpnln_AL_roi_adj.feather'
    if os.path.exists(cache):
        roi = pd.read_feather(cache)
    else:
        from dotenv import load_dotenv
        from neuprint import Client, fetch_adjacencies, set_default_client
        load_dotenv()
        cl = Client('https://neuprint.janelia.org', dataset='male-cns:v1.0', token=os.getenv('NEUPRINT_TOKEN'))
        set_default_client(cl)
        _, roi = fetch_adjacencies(sources=orns['bodyId'].astype(int).tolist(),
                                   targets=postn['bodyId'].astype(int).tolist(),
                                   rois=['AL(L)', 'AL(R)'], client=cl)
        roi.to_feather(cache)
    id2role = meta.set_index('bodyId')['role']
    id2side = meta.set_index('bodyId')['side']
    id2glom = orns.set_index('bodyId')['glomerulus']
    roi = roi.copy()
    roi['kind'] = roi['bodyId_post'].map(id2role).map({'ALPN': 'PN', 'ALLN': 'LN'})
    roi = roi.dropna(subset=['kind'])
    e = pd.DataFrame({
        'glomerulus': roi['bodyId_pre'].map(id2glom).values,
        'orn_side': roi['bodyId_pre'].map(id2side).values,
        'syn_AL': roi['roi'].map({'AL(L)': 'left', 'AL(R)': 'right'}).values,
        'orn_id': roi['bodyId_pre'].values, 'weight': roi['weight'].values, 'kind': roi['kind'].values,
    })
    r = pd.concat([out, e], ignore_index=True).dropna(subset=['glomerulus', 'orn_side', 'syn_AL'])
    r['animal'] = 'MCNS'
    return r

In [ ]:
_NP_SIDE = {'AL_L': 'left', 'AL_R': 'right'}

def _cave_syn(nb, conn_csv, chunksize=None):
    '''Long synapse table for a CAVE-style dataset given neuron metadata `nb`
    (columns id, side, role, glomerulus). Uses the per-edge `neuropil` as the AL the
    synapse sits in. Returns columns: glomerulus, orn_side, syn_AL, orn_id, weight, kind.'''
    nb = nb.copy(); nb['id'] = nb['id'].astype('int64')
    orns = nb[(nb['role'] == 'ORN') & nb['side'].notna() & nb['glomerulus'].notna()]
    orn_ids = set(orns['id'])
    pn = set(nb[nb['role'] == 'ALPN']['id']); ln = set(nb[nb['role'] == 'ALLN']['id'])
    id2side = nb.set_index('id')['side']; id2glom = orns.set_index('id')['glomerulus']
    parts = []
    reader = pd.read_csv(conn_csv, usecols=['pre_root_id', 'post_root_id', 'neuropil', 'syn_count'], chunksize=chunksize)
    for ch in (reader if chunksize else [reader]):
        c = ch[ch['pre_root_id'].isin(orn_ids) & ch['neuropil'].isin(['AL_L', 'AL_R'])]
        if len(c) == 0:
            continue
        c = c.assign(glomerulus=c['pre_root_id'].map(id2glom),
                     orn_side=c['pre_root_id'].map(id2side),
                     syn_AL=c['neuropil'].map(_NP_SIDE))
        o = c.groupby(['glomerulus', 'orn_side', 'syn_AL', 'pre_root_id'], as_index=False)['syn_count'].sum()
        o['kind'] = 'out'
        parts.append(o.rename(columns={'pre_root_id': 'orn_id', 'syn_count': 'weight'}))
        for kind, ids in [('PN', pn), ('LN', ln)]:
            cc = c[c['post_root_id'].isin(ids)]
            if len(cc):
                g = cc.groupby(['glomerulus', 'orn_side', 'syn_AL', 'pre_root_id'], as_index=False)['syn_count'].sum()
                g['kind'] = kind
                parts.append(g.rename(columns={'pre_root_id': 'orn_id', 'syn_count': 'weight'}))
    return pd.concat(parts, ignore_index=True)


def load_fafb():
    '''FAFB/FlyWire (CAVE). PN filter (uniglomerular): classification sub_class == 'uniglomerular'.
    LN filter (GABAergic): cross-references neurons.csv.gz nt_type == 'GABA'.'''
    nb = pd.read_csv(f'{DATA}/FlyWire/classification.csv.gz').rename(columns={'root_id': 'id', 'hemibrain_type': 'type'})
    nb['side'] = nb['side'].map(norm_side); typ = nb['type'].astype(str)

    # -- uniglomerular PN filter (sub_class) ----------------------------------
    is_upn = nb['class'].eq('ALPN') & nb['sub_class'].astype(str).eq('uniglomerular')
    # -- GABAergic LN filter (cross-reference neurons.csv.gz nt_type) ---------
    nt = pd.read_csv(f'{DATA}/FlyWire/neurons.csv.gz', usecols=['root_id', 'nt_type'])
    gaba_ids = set(nt.loc[nt['nt_type'] == 'GABA', 'root_id'].astype('int64'))
    nb['id64'] = nb['id'].astype('int64')
    is_gaba_ln = nb['class'].eq('ALLN') & nb['id64'].isin(gaba_ids)
    nb = nb.drop(columns=['id64'])

    nb['role'] = np.select([typ.map(is_orn), is_upn, is_gaba_ln],
                           ['ORN', 'ALPN', 'ALLN'], default=None)
    nb['glomerulus'] = np.where(nb['role'] == 'ORN', typ.map(glom), None)
    r = _cave_syn(nb, f'{DATA}/FlyWire/connections_princeton_no_threshold.csv.gz', chunksize=3_000_000)
    r['animal'] = 'FAFB'; return r


def load_banc():
    '''BANC (CAVE). PN filter (uniglomerular): Sub Class == 'uniglomerular_projection_neuron'.
    LN filter (GABAergic): Predicted NT type == 'GABA'.'''
    nb = pd.read_csv(f'{DATA}/BANC/neurons.csv.gz').rename(
        columns={'Root ID': 'id', 'Primary Cell Type': 'type', 'Soma side': 'side', 'Class': 'class'})
    nb['side'] = nb['side'].map(norm_side); cls = nb['class'].astype(str)

    # -- uniglomerular PN filter (Sub Class) ----------------------------------
    is_upn = cls.eq('antennal_lobe_projection_neuron') & nb['Sub Class'].astype(str).eq('uniglomerular_projection_neuron')
    # -- GABAergic LN filter (Predicted NT type) ------------------------------
    is_gaba_ln = cls.eq('antennal_lobe_local_neuron') & nb['Predicted NT type'].astype(str).eq('GABA')

    nb['role'] = np.select([cls.eq('olfactory_receptor_neuron'), is_upn, is_gaba_ln],
                           ['ORN', 'ALPN', 'ALLN'], default=None)
    nb['glomerulus'] = np.where(nb['role'] == 'ORN', nb['type'].astype(str).map(glom), None)
    r = _cave_syn(nb, f'{DATA}/BANC/connections_princeton.csv.gz')
    r['animal'] = 'BANC'; return r


def load_hemibrain():
    '''Hemibrain (neuPrint) — a single-hemisphere volume, but its RIGHT AL is complete and
    contains the contralateral (_L) ORN arbors, so the complete right AL gives an unbiased
    contra/ipsi measurement (ipsi = _R ORN, contra = _L ORN). R_contra uses mirror symmetry
    (contra-ORN output into the right AL stands in for right-ORN output into the absent left AL).

    PN filter (uniglomerular): type does NOT start with 'M_' (excludes multiglomerular).
    LN filter (GABAergic): cross-references hemibrain_with_nt mean CSV; max-NT = GABA.
    Cached to data/Hemibrain/orn_lat_syn_upn_gaba.feather; regenerates if missing.'''
    cache = f'{DATA}/Hemibrain/orn_lat_syn_upn_gaba.feather'
    old_cache = f'{DATA}/Hemibrain/orn_lat_syn.feather'
    if os.path.exists(cache):
        return pd.read_feather(cache)
    # fallback: if old cache exists but new one doesn't, try to filter it
    if os.path.exists(old_cache) and os.getenv('NEUPRINT_TOKEN', '').strip() == '':
        print('WARNING: Hemibrain upn-gaba cache missing and no NEUPRINT_TOKEN. '
              'Falling back to OLD cache (unfiltered PNs/LNs). Set NEUPRINT_TOKEN to regenerate.')
        r = pd.read_feather(old_cache)
        return r
    from dotenv import load_dotenv
    from neuprint import Client, fetch_neurons, fetch_adjacencies, set_default_client, NeuronCriteria as NC
    load_dotenv()
    c = Client('https://neuprint.janelia.org', dataset='hemibrain:v1.2.1', token=os.getenv('NEUPRINT_TOKEN'))
    set_default_client(c)
    orn, _ = fetch_neurons(NC(type='ORN_.*', regex=True))
    orn['oside'] = orn['instance'].astype(str).str.extract(r'_(L|R)$')[0].map({'L': 'left', 'R': 'right'})
    orn['glomerulus'] = orn['type'].astype(str).map(glom)
    orn = orn.dropna(subset=['oside', 'glomerulus'])
    af = lambda ri: (ri.get('AL(R)', {}) if isinstance(ri, dict) else {}).get('downstream', 0) or 0
    orn['alR'] = orn['roiInfo'].apply(af)
    out = pd.DataFrame({'glomerulus': orn['glomerulus'].values, 'orn_side': orn['oside'].values,
                        'syn_AL': 'right', 'orn_id': orn['bodyId'].values, 'weight': orn['alR'].values, 'kind': 'out'})
    alr, _ = fetch_neurons(NC(rois=['AL(R)']))
    tt = alr['type'].astype(str)
    alr['kind'] = np.where(tt.str.contains('PN') & ~tt.str.startswith('ORN'), 'PN',
                           np.where(tt.str.contains('LN'), 'LN', None))

    # -- uniglomerular PN filter: exclude M_ prefix types ---------------------
    alr_upn = alr[(alr['kind'] == 'PN') & ~tt.str.startswith('M_')].copy()
    # -- GABAergic LN filter: cross-reference hemibrain_with_nt ---------------
    nt = pd.read_csv(f'{DATA}/Hemibrain/hemibrain_with_nt/traced-neurons_withnt_mean.csv')
    nt_cols = ['nts_8.gaba','nts_8.acetylcholine','nts_8.glutamate',
               'nts_8.serotonin','nts_8.octopamine','nts_8.dopamine']
    gaba_body_ids = set(nt.loc[nt[nt_cols].idxmax(axis=1) == 'nts_8.gaba', 'bodyId'].astype(int))
    alr_gaba = alr[(alr['kind'] == 'LN') & alr['bodyId'].isin(gaba_body_ids)].copy()

    pn = set(alr_upn['bodyId']); ln = set(alr_gaba['bodyId'])
    _, adj = fetch_adjacencies(sources=orn['bodyId'].astype(int).tolist(), targets=list(pn | ln), rois=['AL(R)'], client=c)
    id2k = {**{i: 'PN' for i in pn}, **{i: 'LN' for i in ln}}
    id2s = orn.set_index('bodyId')['oside']; id2g = orn.set_index('bodyId')['glomerulus']
    e = pd.DataFrame({'glomerulus': adj['bodyId_pre'].map(id2g).values, 'orn_side': adj['bodyId_pre'].map(id2s).values,
                      'syn_AL': 'right', 'orn_id': adj['bodyId_pre'].values,
                      'weight': adj['weight'].values, 'kind': adj['bodyId_post'].map(id2k).values})
    r = pd.concat([out, e], ignore_index=True).dropna(subset=['glomerulus', 'orn_side', 'kind'])
    r['animal'] = 'hemibrain'
    r.to_feather(cache)
    return r

## 2. Load all 4 animals → one long synapse table
MCNS pickle/feather are large and hemibrain fetches from neuPrint (cached); first run ~1–2 min.

In [ ]:
syn = pd.concat([load_mcns(), load_fafb(), load_banc(), load_hemibrain()], ignore_index=True)
# series = one complete antennal lobe; ipsi = ORN soma side matches the AL the synapse sits in
syn['series'] = syn['animal'] + '-' + syn['syn_AL'].map({'left': 'L', 'right': 'R'})
syn['ipsi'] = syn['orn_side'] == syn['syn_AL']
syn['metric'] = syn['kind'].map({'out': 'R_contra', 'PN': 'P_PN', 'LN': 'P_LN'})

for a in ['MCNS', 'FAFB', 'BANC', 'hemibrain']:
    s = syn[syn['animal'] == a]
    print(f'{a:>9s}: {s["orn_id"].nunique():>5d} ORNs | series {sorted(s["series"].unique())}')

## 3. Summary table — per (animal, AL-series, glomerulus, metric)
Pooled ipsi / contra synapses and the contra/ipsi ratio for each of the 7 antennal-lobe series.

In [ ]:
# pooled ipsi/contra synapses per (animal, series, glomerulus, metric)
agg = (syn.groupby(['animal', 'series', 'glomerulus', 'metric', 'ipsi'])['weight'].sum()
          .unstack('ipsi', fill_value=0))
for col in [True, False]:
    if col not in agg:
        agg[col] = 0
agg = agg.rename(columns={True: 'ipsi_syn', False: 'contra_syn'}).reset_index()

# n = number of contributing ORNs (from the 'out' rows) per series x glomerulus
norn = (syn[syn['kind'] == 'out'].groupby(['series', 'glomerulus'])['orn_id']
        .nunique().rename('n').reset_index())
summary = agg.merge(norn, on=['series', 'glomerulus'], how='left')
summary['hemisphere'] = summary['series'].str[-1]
summary['ratio'] = cratio(summary['contra_syn'], summary['ipsi_syn'])
summary['log2_ratio'] = log2r(summary['contra_syn'], summary['ipsi_syn'])
summary = summary[['animal', 'series', 'hemisphere', 'glomerulus', 'metric',
                   'ipsi_syn', 'contra_syn', 'n', 'ratio', 'log2_ratio']]
summary.to_csv('ORN-lateralization-multi.csv', index=False)
print('saved ORN-lateralization-multi.csv  ', summary.shape)
summary.head()

## 4. Common glomeruli (present in all 4 animals) + plotting setup
Requiring hemibrain narrows the shared set. Also defines the AL-series order, animal colours, and hemisphere markers.

In [ ]:
# glomeruli present in ALL animals (so hemibrain's ~40 narrows the set)
common = sorted(set.intersection(*summary.groupby('animal')['glomerulus'].agg(set).tolist()))
print(f'{len(common)} glomeruli common to all {summary["animal"].nunique()} animals:')
print(', '.join(common))

ANIMALS = [a for a in ['MCNS', 'FAFB', 'BANC', 'hemibrain'] if a in summary['animal'].unique()]
SERIES_ORDER = [s for s in ['MCNS-L', 'MCNS-R', 'FAFB-L', 'FAFB-R', 'BANC-L', 'BANC-R', 'hemibrain-R']
                if s in summary['series'].unique()]
animal_color = dict(zip(ANIMALS, sns.color_palette('Set2', len(ANIMALS))))
hemi_marker = {'L': 'o', 'R': 's'}   # left = circle, right = square
METRICS = [('R_contra', r'$R_{contra}$'),
           ('P_PN', r'$P_{PN}$ (uniglomerular)'),
           ('P_LN', r'$P_{LN}$ (GABAergic)')]
print('series:', SERIES_ORDER)

## 5. Heatmaps — 7 antennal-lobe series

One panel per metric; rows = glomeruli (shared), columns = the 7 AL-series (animal × hemisphere).
Colour = raw `contra/ipsi`: **blue ≈ 0 = ipsi, white = 1 = balanced, red > 1 = contra**.

In [ ]:
plot = summary[summary['glomerulus'].isin(common)].copy()
rfin = plot['ratio'].replace(np.inf, np.nan)

vmax = float(np.nanquantile(rfin, 0.98))
norm = TwoSlopeNorm(vcenter=1.0, vmin=0.0, vmax=max(vmax, 1.5))
ax_order = (plot[plot['metric'] == 'R_contra']
            .groupby('glomerulus')['ratio'].mean().sort_values(ascending=False).index.tolist())

fig, axes = plt.subplots(1, 3, figsize=(9, 13), sharey=True, dpi=600)
for ax, (metric, title) in zip(axes, METRICS):
    mat = (plot[plot['metric'] == metric]
           .pivot(index='glomerulus', columns='series', values='ratio')
           .reindex(index=ax_order, columns=SERIES_ORDER))
    sns.heatmap(mat, ax=ax, cmap='RdBu_r', norm=norm, cbar=False,
                annot=True, fmt='.2f', annot_kws={'size': 5.5},
                linewidths=0.4, linecolor='0.2', square=False)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.tick_params(axis='x', labelsize=7.5, rotation=90)
    ax.tick_params(axis='y', labelsize=7.5)
axes[0].set_ylabel('glomerulus', fontsize=11)

fig.subplots_adjust(right=0.9, wspace=0.1)
cax = fig.add_axes([0.93, 0.35, 0.018, 0.3])
fig.colorbar(ScalarMappable(norm=norm, cmap='RdBu_r'), cax=cax,
             label='contra / ipsi  (0=ipsi, 1=balanced)')
fig.suptitle('ORN-pathway lateralisation — 7 antennal lobes (4 animals)', y=0.995, fontsize=14)

plt.savefig('ORN-lateralization-multi-heatmap.png', dpi=600, bbox_inches='tight')
plt.show()

## 6. Concordance — ratio per glomerulus across the 7 ALs

Marker per AL-series: **colour = animal**, **shape = hemisphere** (circle = left AL, square = right AL).
0 = ipsi, dashed line at 1 = balanced; red-edged markers are clipped at the cap.

In [ ]:
# per-glomerulus contra/ipsi across the 7 ALs: colour = animal, marker = hemisphere
# (circle = left AL, square = right AL). 0 = ipsi, dashed line at 1 = balanced.
from matplotlib.lines import Line2D
rfin = plot['ratio'].replace(np.inf, np.nan)
cap = float(np.ceil(np.nanquantile(rfin[np.isfinite(rfin)], 0.98) * 2) / 2)
fig, axes = plt.subplots(1, 3, figsize=(15, 11), sharey=True, dpi=600)
for ax, (metric, title) in zip(axes, METRICS):
    d = plot[plot['metric'] == metric].copy()
    d['x'], d['extreme'] = cap_extreme(d['ratio'], cap)
    order = d.groupby('glomerulus')['x'].mean().reindex(ax_order).index
    ypos = {g: i for i, g in enumerate(order)}
    for g in order:
        vals = d[d['glomerulus'] == g]['x']
        ax.plot([vals.min(), vals.max()], [ypos[g], ypos[g]], color='0.4', lw=1, zorder=0)
    for _, row in d.iterrows():
        ax.scatter(row['x'], ypos[row['glomerulus']], s=42, color=animal_color[row['animal']],
                   marker=hemi_marker[row['hemisphere']],
                   edgecolor=('red' if row['extreme'] else 'w'),
                   linewidth=(1.6 if row['extreme'] else 0.3), zorder=2)
    ax.axvline(1.0, color='r', ls='--', lw=1)
    ax.set_xlim(-0.05, cap + 0.1)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('contra / ipsi  (0 = ipsi, 1 = balanced)')
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=7.5)
    ax.set_ylim(-1, len(order)); ax.grid(axis='x', alpha=0.3)
axes[0].set_ylabel('glomerulus', fontsize=11)
handles = [Line2D([], [], marker='o', ls='', color=animal_color[a], label=a, mec='w') for a in ANIMALS]
handles += [Line2D([], [], marker=hemi_marker[h], ls='', color='0.7', mec='w',
                   label={'L': 'left AL', 'R': 'right AL'}[h]) for h in ['L', 'R']]
axes[2].legend(handles=handles, loc='lower right', fontsize=8)
fig.suptitle(f'Contra/ipsi ratio per glomerulus across 7 ALs  (red edge = clipped at {cap:g})',
             y=1.0, fontsize=12)
plt.tight_layout()
plt.savefig('ORN-lateralization-multi-scatter.png', dpi=600, bbox_inches='tight')
plt.show()

## 6b. ⚠ Dataset-quality audit — the MCNS left-AL side labels are not trustworthy

The broad panels above pool all 7 AL-series. A systematic audit of the raw data
(side-label balance, L/R self-consistency, cross-dataset concordance, and
cross-validation against hemibrain's curated instance-name sides) found that one
series — **MCNS-L — is dominated by an artifact, not biology**, and that the
problem leaks into **R_contra, P_PN *and* P_LN**:

* **MCNS ORN soma side comes from `rootSide`** — every ORN soma lies outside the
  CNS volume, so `somaSide` is empty. `rootSide` is unreliable for ORNs: per-type
  L:R ORN counts are imbalanced for 47/53 glomeruli (median ratio 0.79; VM6l 2 vs
  12, VA1v 15 vs 73), whereas FAFB (median 1.00) and BANC (median 1.09) are
  symmetric, as real antennal ORN numbers are.
* Because the **P_PN / P_LN rows are ORN→PN/LN edges tagged with the ORN's side**,
  the same artifact inflates all three metrics in MCNS: a uniform L≫R offset in
  48–49 of 53 glomeruli (vs 26–29/51 in FAFB and 17–24/54 in BANC), and MCNS's
  left and right half-connectomes do not even correlate with each other
  (Spearman ≈ 0 for every metric; FAFB 0.47–0.68, BANC 0.55–0.65).
* **MCNS-L is a global outlier**: its per-glomerulus ratios are uncorrelated with
  every other series (≈ 0 vs BANC/hemibrain), and its left-AL ORN-synapse totals
  are 18–29% low. MCNS-R alone agrees with FAFB / BANC / hemibrain (ρ 0.4–0.6).
* The trustworthy subset for laterality is **FAFB + BANC (+ hemibrain-R as a
  mirror-symmetry cross-check, whose `_L/_R` sides come from curated instance
  names)**. The male-CNS left AL should not be used for ORN-laterality claims.

Because of this, **every figure below (sections 7–12) is repeated in
dataset-specific versions** (MCNS only / FAFB only / hemibrain only / BANC only),
so the reliable signal can be read separately from the MCNS artifact. The
reproducible audit numbers are computed in the cell below.

In [ ]:
# ── Reproducible dataset-quality audit (see section 6b) ─────────────────────
# (1) Per-type ORN count balance by SOMA SIDE (from the raw synapse table `syn`;
#     summary['n'] is L/R-symmetric by construction — every ORN has a row in both ALs).
ocnt = (syn[syn['kind'] == 'out']
        .groupby(['animal', 'glomerulus', 'orn_side'])['orn_id'].nunique()
        .unstack('orn_side', fill_value=0))
ocnt['ratio_LR'] = ocnt['left'] / ocnt['right'].replace(0, np.nan)
nsum = ocnt.groupby('animal')['ratio_LR'].agg(['median', 'count',
        lambda s: int(((s > 1.5) | (s < 0.67)).sum())]).rename(
        columns={'median': 'median L/R', 'count': '# types', '<lambda>': '# imbalanced (>1.5x)'})
print('Per-type ORN count balance by soma side  (real antennal ORN numbers are L/R symmetric):')
print(nsum.to_string())

# (2) L-vs-R self-consistency of the pooled ratio within each animal (Spearman).
print('\nL-vs-R self-consistency of contra/ipsi ratio per glomerulus (Spearman rho):')
selfc = []
for a in ['MCNS', 'FAFB', 'BANC']:
    piv = summary[summary['animal'] == a].pivot_table(
        index=['glomerulus', 'metric'], columns='hemisphere', values='ratio')
    row = {'animal': a}
    for m in ['R_contra', 'P_PN', 'P_LN']:
        tmp = piv.xs(m, level='metric').replace(np.inf, np.nan)
        row[m] = float(tmp['L'].corr(tmp['R'], method='spearman'))
    selfc.append(row)
print(pd.DataFrame(selfc).set_index('animal').round(2).to_string())
print('\nInterpretation: FAFB/BANC are self-consistent (rho > 0.45) with balanced ORN counts;\n'
      'MCNS is not — its ORN side labels (rootSide) and left-AL series are unreliable.')

## 7. Per-ORN spread of axon laterality

Distribution over individual ORNs (each contributes one raw `contra/ipsi` ratio), by
glomerulus, faceted by dataset. 0 = fully ipsi, dashed line at 1 = balanced. Individual
ORNs with `ipsi = 0` give ∞ and are clipped to the cap, marked with a red ✕.

In [ ]:
# per-ORN raw contra/ipsi from the 'out' rows, split by the ORN's hemisphere (soma side).
# hemibrain is excluded here: its left AL is truncated, so a single ORN's contra output
# isn't measurable (the pooled hemibrain-R series in the other panels uses mirror symmetry).
o = syn[(syn['kind'] == 'out') & (syn['animal'] != 'hemibrain') & syn['glomerulus'].isin(common)].copy()
o['role'] = np.where(o['syn_AL'] == o['orn_side'], 'ipsi', 'contra')
po = (o.pivot_table(index=['animal', 'orn_side', 'orn_id', 'glomerulus'],
                    columns='role', values='weight', fill_value=0).reset_index())
for c in ['ipsi', 'contra']:
    if c not in po: po[c] = 0
po['r'] = cratio(po['contra'], po['ipsi'])
po = po.dropna(subset=['r'])
po['hemisphere'] = po['orn_side'].map({'left': 'L', 'right': 'R'})
cap = float(np.ceil(np.nanquantile(po['r'][np.isfinite(po['r'])], 0.98)))
po['x'], po['extreme'] = cap_extreme(po['r'], cap)
order = po.groupby('glomerulus')['r'].median().sort_values(ascending=False).index

full = [a for a in ANIMALS if a != 'hemibrain']
fig, axes = plt.subplots(len(full), 1, figsize=(15, 9), sharex=True, sharey=True, dpi=600)
for ax, a in zip(axes, full):
    dd = po[po['animal'] == a]
    sns.boxplot(data=dd, x='glomerulus', y='x', hue='hemisphere', order=order,
                hue_order=['L', 'R'], ax=ax, fliersize=0, showcaps=False, linewidth=0.7,
                palette={'L': '#5a7da8', 'R': '#a85a7d'})
    ex = dd[dd['extreme']]
    if len(ex):
        sns.stripplot(data=ex, x='glomerulus', y='x', order=order, ax=ax,
                      color='red', marker='X', size=4, jitter=0.2, legend=False)
    ax.axhline(1.0, color='r', lw=1, ls='--')
    ax.set_ylim(-0.2, cap + 0.3)
    ax.set_ylabel(f'{a}\ncontra / ipsi', fontsize=9)
    ax.set_xlabel(''); ax.grid(axis='y', alpha=0.3)
    ax.legend(title='hemisphere', fontsize=7, loc='upper right')
axes[-1].tick_params(axis='x', rotation=90, labelsize=8)
fig.suptitle(f'Per-ORN axon laterality by glomerulus and hemisphere  '
             f'(0 = ipsi, 1 = balanced; red X = clipped at {cap:g})', fontsize=13)
plt.tight_layout()
plt.savefig('ORN-lateralization-multi-box.png', dpi=600, bbox_inches='tight')
plt.show()

> ⚠ **MCNS row caveat.** The left/right (hemisphere) split in the MCNS panel is
> inflated by the male-CNS `rootSide` side-label artifact documented in section 6b:
> ORN soma sides are unreliable when the soma lies outside the volume, so the L/R
> pools are contaminated and the red-X "fully contra" ORNs there are largely
> mislabeled-side cells rather than real bilateral ORNs. The reliable per-ORN
> comparison is **FAFB vs BANC** (hemibrain is excluded by design — its left AL is
> truncated).

## 8. Glomeruli in 3-D lateralisation space

One point per glomerulus = **median across the 7 ALs** of each raw `contra/ipsi` ratio
(**R_contra**, **P_LN**, **P_PN**). **Origin (0,0,0) = fully ipsilateral**; dashed guides at
ratio = 1 (balanced); colour = R_contra; clipped extrema ringed red.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)

# one point per glomerulus = median raw contra/ipsi ratio across the 7 ALs
AXES = [('R_contra', 'R_contra  (ORN axon, contra/ipsi)'),
        ('P_LN', 'P_LN  (ALLN input, contra/ipsi)'),
        ('P_PN', 'P_PN  (ALPN input, contra/ipsi)')]
xm, ym, zm = [m for m, _ in AXES]

sp = summary[summary['glomerulus'].isin(common)].copy()
sp['rf'] = sp['ratio'].replace(np.inf, np.nan)
space = sp.groupby(['glomerulus', 'metric'])['rf'].median().unstack('metric').reset_index()

cap = float(np.ceil(np.nanquantile(space[[xm, ym, zm]].values[np.isfinite(space[[xm, ym, zm]].values)], 0.98)))
extreme = np.zeros(len(space), bool)
for m in (xm, ym, zm):
    space[m], ex = cap_extreme(space[m], cap)
    extreme |= ex

fig = plt.figure(figsize=(11, 9), dpi=600)
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(space[xm], space[ym], space[zm], c=space[xm], cmap='viridis',
                s=55, edgecolor='w', linewidth=0.4, depthshade=False)
if extreme.any():
    e = space[extreme]
    ax.scatter(e[xm], e[ym], e[zm], s=120, facecolors='none', edgecolors='red', linewidth=1.5)
for _, r in space.iterrows():
    ax.text(r[xm], r[ym], r[zm], r['glomerulus'], size=6.5, color='0.8')

ax.plot([1, 1], [0, cap], [0, 0], color='0.5', lw=0.7, ls='--')
ax.plot([0, cap], [1, 1], [0, 0], color='0.5', lw=0.7, ls='--')
ax.plot([0, 0], [1, 1], [0, cap], color='0.5', lw=0.7, ls='--')

ax.set_xlabel(AXES[0][1]); ax.set_ylabel(AXES[1][1]); ax.set_zlabel(AXES[2][1])
ax.set_xlim(0, cap); ax.set_ylim(0, cap); ax.set_zlim(0, cap)
ax.view_init(elev=22, azim=-60)
ax.set_title(f'Glomeruli in lateralisation space  (median over 7 ALs; origin = fully ipsi; '
             f'red ring = clipped at {cap:g})')
fig.colorbar(sc, ax=ax, shrink=0.5, pad=0.1, label='R_contra (ORN axon, contra/ipsi)')
plt.tight_layout()
plt.savefig('ORN-lateralization-multi-3d.png', dpi=600, bbox_inches='tight')
plt.show()

# Tip: run `%matplotlib widget` (needs ipympl) in a cell above for interactive rotation.

### 8b. Dataset-specific lateralisation space

Same axes as section 8, but one panel per animal: each point = median raw
contra/ipsi over **that animal's own** AL-series (single series for hemibrain).
⚠ The MCNS panel inherits the section-6b side-label artifact (its left-AL series
is untrustworthy); read FAFB / BANC / hemibrain panels as the reliable signal.

In [ ]:
# ── Dataset-specific 3-D lateralisation space (one panel per animal) ─────────
fig = plt.figure(figsize=(17, 13), dpi=600)
for i, a in enumerate(ANIMALS, 1):
    ax = fig.add_subplot(2, 2, i, projection='3d')
    sp_a = summary[(summary['animal'] == a) & summary['glomerulus'].isin(common)].copy()
    sp_a['rf'] = sp_a['ratio'].replace(np.inf, np.nan)
    space_a = sp_a.groupby(['glomerulus', 'metric'])['rf'].median().unstack('metric').reset_index()
    vals = space_a[[xm, ym, zm]].values
    cap_a = float(np.ceil(np.nanquantile(vals[np.isfinite(vals)], 0.98))) or 1.0
    extreme_a = np.zeros(len(space_a), bool)
    for m in (xm, ym, zm):
        space_a[m], ex = cap_extreme(space_a[m], cap_a)
        extreme_a |= ex
    sc = ax.scatter(space_a[xm], space_a[ym], space_a[zm], c=space_a[xm],
                    cmap='viridis', s=55, edgecolor='w', linewidth=0.4, depthshade=False)
    if extreme_a.any():
        e = space_a[extreme_a]
        ax.scatter(e[xm], e[ym], e[zm], s=120, facecolors='none', edgecolors='red', linewidth=1.5)
    for _, r in space_a.iterrows():
        ax.text(r[xm], r[ym], r[zm], r['glomerulus'], size=5.5, color='0.8')
    ax.plot([1, 1], [0, cap_a], [0, 0], color='0.5', lw=0.7, ls='--')
    ax.plot([0, cap_a], [1, 1], [0, 0], color='0.5', lw=0.7, ls='--')
    ax.plot([0, 0], [1, 1], [0, cap_a], color='0.5', lw=0.7, ls='--')
    ax.set_xlabel(AXES[0][1], fontsize=7); ax.set_ylabel(AXES[1][1], fontsize=7); ax.set_zlabel(AXES[2][1], fontsize=7)
    ax.set_xlim(0, cap_a); ax.set_ylim(0, cap_a); ax.set_zlim(0, cap_a)
    ax.view_init(elev=22, azim=-60)
    ax.set_title(f'{a} only  (median over its AL-series; red ring = clipped at {cap_a:g})', fontsize=10)
fig.suptitle('Dataset-specific lateralisation space (origin = fully ipsi; 1 = balanced)', y=0.99, fontsize=14)
# plt.tight_layout() 
plt.savefig('ORN-lateralization-multi-3d-by-animal.png', dpi=600, bbox_inches='tight')
plt.show()

## 9. 2-D view: PN vs LN input laterality

**P_PN** (x) vs **P_LN** (y) as raw `contra/ipsi`, one point per glomerulus = **median over the
7 ALs**. **Origin = fully ipsi**, dashed lines at 1 = balanced. Colour = R_contra, size = median
#ORNs; clipped extrema ringed red.

In [ ]:
plt.style.use('dark_background')
# 2-D view: P_PN (x) vs P_LN (y), one point per glomerulus = median raw contra/ipsi over the 7 ALs.
# 0 = fully ipsi (origin), 1 = balanced (dashed), >1 = contra. colour = R_contra, size = median #ORNs.
# Error bars = IQR (25th-75th pct) across the 7 AL-series, i.e. the spread of datasets around the
# median. Robust to the skew/extremes of the ratio; descriptive only (the 7 ALs are pseudo-replicates).
sp = summary[summary['glomerulus'].isin(common)].copy()
sp['rf'] = sp['ratio'].replace(np.inf, np.nan)
piv = sp.groupby(['glomerulus', 'metric'])['rf'].median().unstack('metric')
piv['n'] = sp[sp['metric'] == 'R_contra'].groupby('glomerulus')['n'].median()
piv = piv.reset_index()

cap = float(np.ceil(np.nanquantile(piv[['R_contra', 'P_PN', 'P_LN']].values[
    np.isfinite(piv[['R_contra', 'P_PN', 'P_LN']].values)], 0.98)))
extreme = np.zeros(len(piv), bool)
for c in ['R_contra', 'P_PN', 'P_LN']:
    piv[c], ex = cap_extreme(piv[c], cap)
    extreme |= ex

def iqr_err(metric):
    '''Asymmetric [lower, upper] whisker lengths from the IQR of the raw ratio across the 7 ALs,
    relative to the (capped) median; endpoints clipped to [0, cap] so they stay on-axis.'''
    g = sp[sp['metric'] == metric].groupby('glomerulus')['rf']
    center = piv[metric].to_numpy()
    q25 = g.quantile(0.25).reindex(piv['glomerulus']).to_numpy()
    q75 = g.quantile(0.75).reindex(piv['glomerulus']).to_numpy()
    q25 = np.where(np.isfinite(q25), np.clip(q25, 0, cap), center)
    q75 = np.where(np.isfinite(q75), np.clip(q75, 0, cap), center)
    return np.vstack([np.clip(center - q25, 0, None), np.clip(q75 - center, 0, None)])

xerr = iqr_err('P_PN')
yerr = iqr_err('P_LN')

smin, smax = 30, 320
nn = piv['n'].fillna(0)
sizes = smin + (smax - smin) * (np.sqrt(nn) - np.sqrt(nn).min()) / (np.sqrt(nn).max() - np.sqrt(nn).min())

fig, ax = plt.subplots(figsize=(9, 7.5), dpi=600)
ax.errorbar(piv['P_PN'], piv['P_LN'], xerr=xerr, yerr=yerr, fmt='none',
            ecolor='0.55', elinewidth=0.8, capsize=2, alpha=0.6, zorder=1)
sc = ax.scatter(piv['P_PN'], piv['P_LN'], c=piv['R_contra'], cmap='viridis',
                s=sizes, edgecolor='w', linewidth=0.4, zorder=2)
ax.scatter(piv.loc[extreme, 'P_PN'], piv.loc[extreme, 'P_LN'],
           s=sizes[extreme] + 40, facecolors='none', edgecolors='red', linewidth=1.6, zorder=3)
for _, r in piv.iterrows():
    ax.annotate(r['glomerulus'], (r['P_PN'], r['P_LN']), fontsize=9,
                xytext=(3, 3), textcoords='offset points', color='white', zorder=4)
ax.axhline(1, color='r', lw=1, ls='--'); ax.axvline(1, color='r', lw=1, ls='--')
ax.set_xlim(-0.05, 1.5); ax.set_ylim(-0.05, cap + 0.1)
ax.set_xlabel('P_PN  (ALPN input, contra/ipsi)'); ax.set_ylabel('P_LN  (ALLN input, contra/ipsi)')
ax.set_title(f'PN vs LN input laterality per glomerulus  '
             f'(median over 7 ALs, error bars = IQR across ALs; 0 = ipsi, 1 = balanced; '
             f'red ring = clipped at {cap:g})', fontsize=11)
fig.colorbar(sc, ax=ax, label='R_contra (ORN axon, contra/ipsi)')

for nref in [10, 50, 150]:
    sref = smin + (smax - smin) * (np.sqrt(nref) - np.sqrt(nn).min()) / (np.sqrt(nn).max() - np.sqrt(nn).min())
    ax.scatter([], [], s=sref, c='0.6', edgecolor='w', linewidth=0.4, label=f'{nref}')
ax.legend(title='# ORNs (median)', loc='upper right', labelspacing=1.1, frameon=True, fontsize=8)

# plot a correlation line
corr = piv[['P_PN', 'P_LN']].corr().iloc[0, 1]
ax.text(1.05, cap, f'r = {corr:.2f}', fontsize=10, color='black')
m, b = np.polyfit(piv['P_PN'], piv['P_LN'], 1)
xvals = np.array(ax.get_xlim())
ax.plot(xvals, m * xvals + b, color='white', lw=1, ls='--', zorder=0)

plt.tight_layout()
plt.savefig('ORN-lateralization-multi-scatter-PN-LN.png', dpi=600, bbox_inches='tight')
plt.show()

### 9b. Dataset-specific PN-vs-LN view

Same 2-D view as section 9, but per animal: each point = median raw contra/ipsi
over **that animal's own** AL-series (single series for hemibrain), coloured by
its R_contra. ⚠ The MCNS panel inherits the section-6b side-label artifact;
FAFB / BANC / hemibrain are the reliable panels.

In [ ]:
# ── Dataset-specific 2-D view: P_PN vs P_LN (one panel per animal) ───────────
fig, axes = plt.subplots(2, 2, figsize=(13, 11), squeeze=False, sharex=True, sharey=True,dpi=600)
for ax, a in zip(axes.ravel(), ANIMALS):
    sp_a = summary[(summary['animal'] == a) & summary['glomerulus'].isin(common)].copy()
    sp_a['rf'] = sp_a['ratio'].replace(np.inf, np.nan)
    piv_a = sp_a.groupby(['glomerulus', 'metric'])['rf'].median().unstack('metric').reset_index()
    vals = piv_a[['R_contra', 'P_PN', 'P_LN']].values
    cap_a = float(np.ceil(np.nanquantile(vals[np.isfinite(vals)], 0.98))) or 1.0
    extreme_a = np.zeros(len(piv_a), bool)
    for c in ['R_contra', 'P_PN', 'P_LN']:
        piv_a[c], ex = cap_extreme(piv_a[c], cap_a)
        extreme_a |= ex
    sc = ax.scatter(piv_a['P_PN'], piv_a['P_LN'], c=piv_a['R_contra'], cmap='viridis',
                    s=65, edgecolor='w', linewidth=0.4)
    ax.scatter(piv_a.loc[extreme_a, 'P_PN'], piv_a.loc[extreme_a, 'P_LN'],
               s=105, facecolors='none', edgecolors='red', linewidth=1.6)
    for _, r in piv_a.iterrows():
        ax.annotate(r['glomerulus'], (r['P_PN'], r['P_LN']), fontsize=7.5,
                    xytext=(3, 3), textcoords='offset points', color='white')
    ax.axhline(1, color='r', lw=1, ls='--'); ax.axvline(1, color='r', lw=1, ls='--')
    ax.set_xlim(-0.05, max(cap_a, 1.1)); ax.set_ylim(-0.05, cap_a + 0.1)
    ax.set_title(f'{a} only  (red ring = clipped at {cap_a:g})', fontsize=10)
    ax.set_xlabel(r'$P_{PN}$  (ALPN input, contra/ipsi)'); ax.set_ylabel(r'$P_{LN}$  (ALLN input, contra/ipsi)')
fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.85, label='R_contra (ORN axon, contra/ipsi)')
fig.suptitle('Dataset-specific PN vs LN input laterality (0 = ipsi, 1 = balanced)', fontsize=13)
plt.savefig('ORN-lateralization-multi-2D.png', dpi=600, bbox_inches='tight')
# plt.tight_layout()
plt.show()

## 10. Depth-slice reveal videos — consensus ALs coloured by lateralization

Three videos (one per metric: **R_contra**, **P_LN**, **P_PN**) of the bilateral
**consensus** antennal lobes placed in a common space (MCNS on the left, FAFB on
the right). A cutting plane sweeps **front→back along Y**; each frame is a
**solid filled cross-section** — the front-most visible surface of everything
*behind* the plane — and each visible glomerulus is **labelled at the centre of
its exposed front face**, so you can read which glomerulus is at the revealing
edge.

- **MCNS (left AL)** glomeruli are coloured by **male** values (`animal == MCNS`);
  **FAFB (right AL)** glomeruli by the **non-male/female** pool
  (FAFB + BANC + hemibrain), median per glomerulus. Colour = the raw contra/ipsi
  `ratio` (this notebook's convention; `inf` dropped), shared 2–98% scale per
  video. Glomeruli with no value render grey.

**Prerequisite:** the consensus AL geometry is built in `volume.ipynb`. Run its
bilateral-consensus + hull-export cells first; this section *loads* the saved
exclusive voxel owner-grids from `glomerulus_output/consensus_hull_meshes/`
(`consensus_voxel_grid_{FAFB,MCNS}.npz`) rather than recomputing them.

Outputs: `slice_reveal_R_contra.mp4`, `slice_reveal_P_LN.mp4`,
`slice_reveal_P_PN.mp4` in `glomerulus_output/`. Reveal axis (`VID_SLICE_AXIS`)
and direction (`VID_FRONT`), labels (`VID_LABELS`/`VID_LABEL_FS`), resolution and
colormap are tunable via the `VID_*` constants.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DEPTH-SLICE REVEAL VIDEOS — consensus AL volumes coloured by lateralization
# ══════════════════════════════════════════════════════════════════════════════
# A cutting plane sweeps the common-space consensus antennal lobes front→back
# along the reveal axis (Y).  Each frame is a SOLID filled cross-section: the
# front-most surface of everything BEHIND the plane, projected onto the plane ⟂
# to the reveal axis, so the cut face points at the viewer.  Each visible
# glomerulus is labelled at the centre of its exposed front face.
#
#   · MCNS-left  glomeruli → coloured by MALE values   (animal == MCNS).
#   · FAFB-right glomeruli → coloured by NON-MALE/female values
#                            (FAFB + BANC + hemibrain), median per glomerulus.
#   · Colour = raw contra/ipsi ratio (this notebook's `ratio`; inf dropped).
#
# The consensus AL geometry is produced by ``volume.ipynb`` (run its bilateral-
# consensus + hull-export cells first).  This loads the saved exclusive voxel
# owner-grids from ``glomerulus_output/consensus_hull_meshes/`` — it does NOT
# recompute the consensus.
from pathlib import Path
import matplotlib
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import matplotlib.animation as animation
import matplotlib.patheffects as pe

# ── Video config ──────────────────────────────────────────────────────────────
VID_METRICS       = ['R_contra', 'P_LN', 'P_PN']
VID_OUT_DIR       = Path('glomerulus_output')
VID_CONSENSUS_DIR = VID_OUT_DIR / 'consensus_hull_meshes'
VID_N_FRAMES      = 90
VID_FPS           = 20
VID_CMAP          = 'viridis'
VID_GREY          = (0.6, 0.6, 0.6, 1.0)   # glomeruli with no metric value
VID_SLICE_AXIS    = 2        # reveal axis: 0=X, 1=Y, 2=Z (image shown across it)
VID_FRONT         = 'low'    # 'low' sweeps from low-coordinate end first; 'high' to flip
VID_LABELS        = True
VID_LABEL_FS      = 10

_VID_METRIC_LABEL = {
    'R_contra': 'R_contra  (contra / ipsi, all targets)',
    'P_LN':     'P_LN  (contra / ipsi onto LNs)',
    'P_PN':     'P_PN  (contra / ipsi onto PNs)',
}


def lateralization_tables():
    """Per-glomerulus contra/ipsi `ratio` (inf→NaN), median over the relevant ALs.
    Uses the in-memory ``summary`` if present, else the saved CSV.  Returns
    (male_tbl, female_tbl) indexed by glomerulus with R_contra/P_LN/P_PN columns."""
    df = globals().get('summary')
    if df is None:
        df = pd.read_csv('ORN-lateralization-multi.csv')
    df = df.copy(); df['rf'] = df['ratio'].replace(np.inf, np.nan)
    male   = df[df['animal'] == 'MCNS']           # male connectome
    female = df[df['animal'] != 'MCNS']           # FAFB + BANC + hemibrain
    mt = male.groupby(['glomerulus', 'metric'])['rf'].median().unstack('metric')
    ft = female.groupby(['glomerulus', 'metric'])['rf'].median().unstack('metric')
    return mt, ft


def load_consensus_grid(consensus_dir: Path = VID_CONSENSUS_DIR) -> dict:
    """Load the two per-dataset exclusive voxel owner-grids saved by volume.ipynb
    (in common FAFB space) and merge them onto one shared grid keyed by (dataset,
    glomerulus).  Returns dict(owner, lo, voxel_nm, dims, keys).

    npz files are produced locally by volume.ipynb (trusted self-generated data);
    allow_pickle is needed only for the string `gloms` array."""
    consensus_dir = Path(consensus_dir)
    parts = {}
    for ds in ('FAFB', 'MCNS'):
        fp = consensus_dir / f'consensus_voxel_grid_{ds}.npz'
        if not fp.exists():
            raise FileNotFoundError(
                f'{fp} not found — run volume.ipynb (bilateral-consensus + '
                f'hull-export cells) first to produce the consensus geometry.')
        z = np.load(fp, allow_pickle=True)
        parts[ds] = dict(owner=z['owner'], lo=np.asarray(z['origin_nm'], float),
                         vnm=float(z['voxel_nm']), gloms=list(z['gloms']))
    vnm = parts['FAFB']['vnm']
    keys = ([('FAFB', g) for g in parts['FAFB']['gloms']] +
            [('MCNS', g) for g in parts['MCNS']['gloms']])
    kindex = {k: i for i, k in enumerate(keys)}
    # world coords + combined key index for every owned voxel in each grid
    pieces = []
    for ds in ('FAFB', 'MCNS'):
        gd = parts[ds]; occ = np.argwhere(gd['owner'] >= 0)
        world = gd['lo'] + (occ + 0.5) * gd['vnm']
        local = gd['owner'][occ[:, 0], occ[:, 1], occ[:, 2]]
        comb = np.array([kindex[(ds, gd['gloms'][o])] for o in local], np.int32)
        pieces.append((world, comb))
    allw = np.vstack([w for w, _ in pieces])
    lo = allw.min(0) - vnm
    dims = np.ceil((allw.max(0) + vnm - lo) / vnm).astype(int)
    owner = np.full(dims, -1, np.int32)
    for world, comb in pieces:
        idx = np.floor((world - lo) / vnm).astype(int)
        owner[idx[:, 0], idx[:, 1], idx[:, 2]] = comb
    return dict(owner=owner, lo=lo, voxel_nm=vnm, dims=dims, keys=keys)


def render_slice_reveal(metric, grid, mt, ft, out_path,
                        n_frames=VID_N_FRAMES, fps=VID_FPS, cmap=VID_CMAP,
                        axis=VID_SLICE_AXIS, front=VID_FRONT,
                        labels=VID_LABELS, label_fs=VID_LABEL_FS):
    """Render one solid cross-section reveal video for `metric`.

    Each frame is the front-most visible surface of all voxels BEHIND the sweep
    plane, projected onto the plane ⟂ to `axis` (a filled image, no gaps).  Each
    visible glomerulus is labelled at the centre of its exposed front face."""
    owner, lo, vnm, keys = grid['owner'], grid['lo'], grid['voxel_nm'], grid['keys']
    nkeys = len(keys); gnames = [g for (_, g) in keys]

    # per-key scalar (MCNS→male table, FAFB→female table) → colour lookup table
    cm_ = matplotlib.colormaps[cmap]
    valk = np.array([
        (mt if ds == 'MCNS' else ft)[metric].get(g, np.nan)
        if metric in (mt if ds == 'MCNS' else ft).columns else np.nan
        for (ds, g) in keys])
    counts = np.bincount(owner[owner >= 0].ravel(), minlength=nkeys)
    vox_vals = np.repeat(valk, counts); fin = np.isfinite(vox_vals)
    vmin, vmax = np.nanpercentile(vox_vals[fin], [2, 98]); norm = Normalize(vmin, vmax)
    lut = np.zeros((nkeys + 1, 4))                       # row 0 → empty (transparent)
    for ki, v in enumerate(valk):
        lut[ki + 1] = cm_(norm(v)) if np.isfinite(v) else VID_GREY

    # image plane = the two non-reveal axes (kept in original order)
    other = [k for k in range(3) if k != axis]; ha, va = other
    n_ax = owner.shape[axis]
    occ_m = np.moveaxis(owner >= 0, axis, 0)             # (n_ax, d_ha, d_va)
    own_m = np.moveaxis(owner, axis, 0)
    if front == 'high':
        occ_m, own_m = occ_m[::-1], own_m[::-1]
    frames_idx = np.linspace(0, n_ax - 1, n_frames).astype(int)
    ext = [lo[ha], lo[ha] + owner.shape[ha] * vnm, lo[va], lo[va] + owner.shape[va] * vnm]
    axname = {0: 'X', 1: 'Y', 2: 'Z'}

    fig = plt.figure(figsize=(12, 6))
    ax  = fig.add_axes([0.02, 0.04, 0.84, 0.9])
    cax = fig.add_axes([0.89, 0.30, 0.02, 0.4])
    fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cm_), cax=cax
                 ).set_label(_VID_METRIC_LABEL.get(metric, metric))

    def draw(i):
        ax.cla()
        pidx = frames_idx[i]
        s = occ_m[pidx:]; has = s.any(0); absidx = pidx + s.argmax(0)
        kp = np.where(has, np.take_along_axis(own_m, absidx[None], axis=0)[0], -1)
        ax.imshow(np.transpose(lut[kp + 1], (1, 0, 2)), origin='lower', extent=ext,
                  interpolation='nearest', aspect='equal')
        p_nm = lo[axis] + (pidx if front == 'low' else (n_ax - 1 - pidx)) * vnm
        if labels:
            # label each visible glomerulus at the centre of its exposed front face
            for ki in np.unique(kp[kp >= 0]):
                ii, jj = np.nonzero(kp == ki)
                ax.text(lo[ha] + (ii.mean() + 0.5) * vnm,
                        lo[va] + (jj.mean() + 0.5) * vnm, gnames[ki],
                        fontsize=label_fs, ha='center', va='center', color='white',
                        path_effects=[pe.withStroke(linewidth=1.8, foreground='black')])
        ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); ax.axis('off')
        ax.set_title(f'{metric}: reveal along {axname[axis]}   '
                     f'({axname[axis]}={p_nm:,.0f} nm,  frame {i + 1}/{n_frames})')
        return ()

    anim = animation.FuncAnimation(fig, draw, frames=n_frames, blit=False)
    try:
        anim.save(str(out_path), writer=animation.FFMpegWriter(fps=fps, bitrate=4000), dpi=120)
        kind = 'mp4'
    except Exception as exc:                                  # no ffmpeg → GIF fallback
        out_path = out_path.with_suffix('.gif')
        anim.save(str(out_path), writer=animation.PillowWriter(fps=fps), dpi=100)
        kind = f'gif (ffmpeg unavailable: {type(exc).__name__})'
    plt.close(fig)
    print(f'    {metric:9s} → {out_path.name}  [{kind}]  (colour range {vmin:.3f}–{vmax:.3f})')
    return out_path


def make_slice_reveal_videos(metrics=VID_METRICS, out_dir: Path = VID_OUT_DIR) -> pd.DataFrame:
    """Load consensus geometry + lateralization values, render one solid
    cross-section reveal video per metric into ``out_dir``.  Returns a manifest."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    mt, ft = lateralization_tables()
    grid = load_consensus_grid()
    print(f'  consensus grid {tuple(grid["dims"])}, {len(grid["keys"])} (dataset, glom) regions, '
          f'{int((grid["owner"] >= 0).sum()):,} voxels')
    print(f'  lateralization: male {mt.shape}, female {ft.shape}')
    rows = []
    for m in metrics:
        path = render_slice_reveal(m, grid, mt, ft, out_dir / f'slice_reveal_{m}.mp4')
        rows.append(dict(metric=m, file=path.name,
                         n_male_gloms=int(mt[m].notna().sum()) if m in mt.columns else 0,
                         n_female_gloms=int(ft[m].notna().sum()) if m in ft.columns else 0))
    return pd.DataFrame(rows)

In [ ]:
# Render the three depth-slice reveal videos into glomerulus_output/.
# (Needs volume.ipynb's consensus geometry on disk — see the cell above.)
slice_videos = make_slice_reveal_videos()
slice_videos

## 11. Functional DSI vs connectomic lateralization (Santana et al.)

Newly published functional-imaging **directional selectivity index (DSI)** per glomerulus
(`data/Santana/glomerulus_data.csv`) compared against this notebook's three connectomic
contra/ipsi ratios — **R_contra** (ORN axon), **P_PN** (ALPN input), **P_LN** (ALLN input).

The Santana imaging is **female**, so the connectomic side uses the **female pool only**
(FAFB + BANC + hemibrain, `animal != 'MCNS'` — the same split as §10's `lateralization_tables()`),
taken as the **median contra/ipsi ratio per glomerulus across the female ALs** (`inf`→NaN, same
ratio convention as §8–§9). One point per glomerulus, coloured by odour type.

Each panel reports **Pearson r** (linear association) and **Spearman ρ** (rank/monotonic
association), both with two-sided p-values. The DSI set is not restricted to the 4-animal
`common` glomeruli; *n* per panel is the number of gloms with both DSI and that metric.

In [ ]:
# --- functional DSI (Santana et al.) vs connectomic contra/ipsi ratios -------
# DSI = imaging-derived directional selectivity index, one value per glomerulus.
# The Santana imaging is FEMALE, so compare against the FEMALE connectome pool only
# (FAFB + BANC + hemibrain, i.e. animal != 'MCNS' — same split as lateralization_tables()).
dsi = pd.read_csv(f'{DATA}/Santana/glomerulus_data.csv')
dsi = dsi.rename(columns={'Glomerulus': 'glomerulus'})[['glomerulus', 'DSI', 'Odor_type']]

# Connectomic side: median raw contra/ipsi ratio per glomerulus across the female ALs (inf->NaN),
# NOT restricted to the 4-animal `common` set so the full DSI glomerulus set can be matched.
_df = globals().get('summary')
if _df is None:
    _df = pd.read_csv('ORN-lateralization-multi.csv')
_df = _df[_df['animal'] != 'MCNS'].copy()          # female connectomes: FAFB + BANC + hemibrain
_df['rf'] = _df['ratio'].replace(np.inf, np.nan)
conn = (_df.groupby(['glomerulus', 'metric'])['rf'].median()
           .unstack('metric')[['R_contra', 'P_PN', 'P_LN']])

dsi_conn = dsi.merge(conn, on='glomerulus', how='left').set_index('glomerulus')
matched = dsi_conn[['R_contra', 'P_PN', 'P_LN']].notna().all(axis=1)
print(f'{matched.sum()} / {len(dsi_conn)} DSI glomeruli matched to all 3 female-connectome metrics')
missing = dsi_conn.index[~matched].tolist()
if missing:
    print('  no connectome match:', ', '.join(missing))
dsi_conn.sort_values('DSI', ascending=False)

In [ ]:
# DSI vs each connectomic contra/ipsi ratio — Pearson (linear) + Spearman (rank).
from scipy import stats

plt.style.use('dark_background')
DSI_METRICS = [('R_contra', r'$R_{contra}$', r'$R_{contra}$  (ORN axon, contra/ipsi)'),
               ('P_PN',     r'$P_{PN}$',     r'$P_{PN}$  (ALPN input, contra/ipsi)'),
               ('P_LN',     r'$P_{LN}$',     r'$P_{LN}$  (ALLN input, contra/ipsi)')]
otypes = list(dsi_conn['Odor_type'].dropna().unique())
opal = dict(zip(otypes, sns.color_palette('Set2', len(otypes))))

fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
for ax, (m, short, xlab) in zip(axes, DSI_METRICS):
    d = dsi_conn[['DSI', m, 'Odor_type']].dropna()
    x, y = d[m].to_numpy(float), d['DSI'].to_numpy(float)
    for ot in otypes:
        s = d[d['Odor_type'] == ot]
        ax.scatter(s[m], s['DSI'], color=opal[ot], s=55, edgecolor='w', linewidth=0.4,
                   label=ot, zorder=2)
    for g, row in d.iterrows():
        ax.annotate(g, (row[m], row['DSI']), fontsize=7, color='0.85',
                    xytext=(3, 3), textcoords='offset points', zorder=3)
    # correlations (two-sided) + least-squares guide line
    pr, pp = stats.pearsonr(x, y)
    sr, sp = stats.spearmanr(x, y)
    slope, intercept = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, slope * xs + intercept, color='white', lw=1, ls='--', alpha=0.7, zorder=1)
    ax.set_xlabel(xlab)
    ax.set_title(f'{short} vs DSI   (n = {len(d)})\n'
                 f'Pearson r = {pr:.2f} (p = {pp:.3g})    Spearman ρ = {sr:.2f} (p = {sp:.3g})',
                 fontsize=9.5)
axes[0].set_ylabel('DSI  (functional directional selectivity)')
axes[0].legend(title='odour type', fontsize=8, title_fontsize=9, loc='best', framealpha=0.3)
fig.suptitle('Functional DSI (Santana et al.) vs connectomic ORN-pathway lateralization',
             y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

### 11b. Dataset-specific: DSI vs connectomic lateralization

The broad panel above compares DSI against the **female connectome pool**
(FAFB + BANC + hemibrain). Here the same correlations are computed against
**each animal's own** pooled contra/ipsi ratio (rows = animal, columns =
metric). ⚠ DSI is measured in females, so the male (MCNS) row is included for
completeness but carries the section-6b side-label artifact.

In [ ]:
# ── Dataset-specific: DSI vs connectomic contra/ipsi (rows = animal, cols = metric) ──
fig, axes = plt.subplots(len(ANIMALS), 3, figsize=(15, 2.9 * len(ANIMALS) + 1.2), squeeze=False)
for i, a in enumerate(ANIMALS):
    _s = summary[summary['animal'] == a].copy()
    _s['rf'] = _s['ratio'].replace(np.inf, np.nan)
    _conn_a = _s.groupby(['glomerulus', 'metric'])['rf'].median().unstack('metric')[['R_contra', 'P_PN', 'P_LN']]
    _d = dsi.merge(_conn_a, on='glomerulus', how='left')
    for j, (m, short, xlab) in enumerate(DSI_METRICS):
        ax = axes[i][j]
        dd = _d[['DSI', m, 'Odor_type']].dropna()
        if len(dd) < 4:
            ax.text(0.5, 0.5, f'no match (n={len(dd)})', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
            continue
        x, y = dd[m].to_numpy(float), dd['DSI'].to_numpy(float)
        for ot in otypes:
            s = dd[dd['Odor_type'] == ot]
            ax.scatter(s[m], s['DSI'], color=opal[ot], s=45, edgecolor='w', linewidth=0.4,
                       label=ot if i == 0 else None, zorder=2)
        pr, pp = stats.pearsonr(x, y); sr, sp = stats.spearmanr(x, y)
        slope, intercept = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 50)
        ax.plot(xs, slope * xs + intercept, color='white', lw=1, ls='--', alpha=0.7, zorder=1)
        ax.set_title(f'{a} · {short}   (n = {len(dd)})\n'
                     f'Pearson r = {pr:.2f} (p = {pp:.3g})    Spearman ρ = {sr:.2f} (p = {sp:.3g})', fontsize=8.5)
        ax.set_xlabel(xlab if i == len(ANIMALS) - 1 else '')
        if j == 0:
            ax.set_ylabel('DSI  (functional directional selectivity)')
axes[0][0].legend(title='odour type', fontsize=7.5, title_fontsize=8, loc='best', framealpha=0.3)
fig.suptitle('Dataset-specific: functional DSI (Santana et al.) vs connectomic ORN-pathway lateralization',
             y=1.0, fontsize=13)
plt.tight_layout()
plt.show()

## 12. Measured DSI vs pre-registered AL-model predictions

Before the DSI data existed we ran a **bilateral AL-model parameter sweep**
(`AL-model/results/hemi_asym/`) scoring how well a GRU decoder reads odour information off PN
spikes across a grid of contralateral-wiring knobs — **R_contra**, **P_pn**, **P_ln**. Those
knobs are *exactly* this notebook's connectomic contra/ipsi ratios (**R_contra**, **P_PN**,
**P_LN**), so we can place each real glomerulus in the model grid and read off the model's
prediction, then relate it to the now-measured **DSI**.

**Method.** Structural condition only (`lateral_gain_norm == True`, GRU decoder, mean over the
10 seeds — per the model's DATA_README, the gain-fixed condition isolates the *structural* effect
of connectivity). For each DSI-analogous **lateral / side-discrimination readout** we build a
`RegularGridInterpolator` over `(R_contra, P_pn, log1p P_ln)` and sample it at each glomerulus's
**female-pool** median coordinates (§11):

| model readout | what it decodes | DSI relation |
|---|---|---|
| `source_side` (AUROC) | which antenna is more strongly driven | most direct directional-selectivity analogue |
| `log_ratio_conc` (R²) | `log(gpL/gpR)`, scale-invariant lateral | scale-invariant lateral coding |
| `contrast_index` (R²) | `(gpL−gpR)/(gpL+gpR)` | scale-invariant lateral coding |
| `difference_conc` (R²) | absolute `gpL−gpR` | absolute lateral coding |

**R_contra axis.** The grid was simulated at `R_contra ∈ {0.5, 1, 2}`, but real values run
0.07–1.04 (many below 0.5). We add an **R_contra = 0 anchor = the unilateral baseline**
(`P_pn = P_ln = 0`, where R_contra is irrelevant because there is no contra drive — verified
~constant across the simulated R_contra): as contra influence vanishes every glomerulus collapses
to that baseline. With this anchor all real R_contra values sit *inside* `[0, 2]`, so no
extrapolation is needed. `P_ln` is log-spaced, so it is interpolated in `log1p` space; `P_pn` is
linear. Query coordinates are clipped to the grid extent.


In [ ]:
# --- Relate the pre-registered AL-model grid predictions to measured DSI --------------------
# The hemispheric-asymmetry sweep (AL-model/results/hemi_asym) scored how well a GRU decoder
# reads odour information off PN spikes across a grid of contra-wiring knobs (R_contra, P_pn,
# P_ln).  Those knobs ARE this notebook's connectomic contra/ipsi ratios (R_contra, P_PN, P_LN),
# so we interpolate the grid at each glomerulus's REAL female-pool coordinates to get a model-
# predicted score, then (next cell) correlate that prediction with the measured DSI.
from scipy.interpolate import RegularGridInterpolator

GRID_PARQUET = '../../AL-model/results/hemi_asym/hemi_sweep_asym.parquet'

# lateral / side-discrimination readouts (the DSI analogues): name -> (regime, readout, metric)
MODEL_READOUTS = {
    'source_side':     ('single_trial', 'source_side',     'auroc'),   # which antenna is stronger
    'log_ratio_conc':  ('single_trial', 'log_ratio_conc',  'r2'),      # scale-invariant lateral
    'contrast_index':  ('single_trial', 'contrast_index',  'r2'),      # scale-invariant lateral
    'difference_conc': ('single_trial', 'difference_conc', 'r2'),      # absolute lateral diff
}

# real per-glomerulus coordinates = female-pool median contra/ipsi ratios (reuse section 11, else rebuild)
if 'dsi_conn' in globals() and {'R_contra', 'P_PN', 'P_LN'}.issubset(dsi_conn.columns):
    coords = dsi_conn.dropna(subset=['R_contra', 'P_PN', 'P_LN']).copy()
else:
    _s = globals().get('summary')
    if _s is None:
        _s = pd.read_csv('ORN-lateralization-multi.csv')
    _s = _s[_s['animal'] != 'MCNS'].copy(); _s['rf'] = _s['ratio'].replace(np.inf, np.nan)
    _conn = (_s.groupby(['glomerulus', 'metric'])['rf'].median()
               .unstack('metric')[['R_contra', 'P_PN', 'P_LN']])
    _dsi = pd.read_csv(f'{DATA}/Santana/glomerulus_data.csv').rename(columns={'Glomerulus': 'glomerulus'})
    coords = (_dsi.merge(_conn, on='glomerulus', how='left')
                  .dropna(subset=['R_contra', 'P_PN', 'P_LN']).set_index('glomerulus'))

# model grid: structural (gain-fixed) condition + GRU decoder
mg = pd.read_parquet(GRID_PARQUET)
mg = mg[(mg['decoder'] == 'gru') & (mg['lateral_gain_norm'] == True)]
Ppn_ax = np.array(sorted(mg['P_pn'].unique()))
Pln_ax = np.array(sorted(mg['P_ln'].unique()))
R_ax   = np.array([0.0, 0.5, 1.0, 2.0])      # 0 = unilateral baseline anchor (no contra drive)
tln_ax = np.log1p(Pln_ax)                    # P_ln is log-spaced -> interpolate in log1p space

# query points: real coords mapped R_contra->R_contra, P_PN->P_pn, P_LN->P_ln, clipped in-grid
qR   = np.clip(coords['R_contra'].to_numpy(float),        R_ax.min(),   R_ax.max())
qPpn = np.clip(coords['P_PN'].to_numpy(float),            Ppn_ax.min(), Ppn_ax.max())
qtln = np.clip(np.log1p(coords['P_LN'].to_numpy(float)),  tln_ax.min(), tln_ax.max())
query = np.column_stack([qR, qPpn, qtln])

def model_interpolator(regime, readout, metric):
    """RegularGridInterpolator over (R_contra, P_pn, log1p P_ln) for one model readout,
    averaged over the 10 seeds.  The R_contra=0 slice is the unilateral baseline
    (P_pn=P_ln=0, ~constant over R_contra): as contra drive vanishes every glomerulus
    collapses to it, which anchors the axis below the simulated minimum of 0.5."""
    d = mg[(mg['regime'] == regime) & (mg['readout'] == readout) & (mg['metric'] == metric)]
    g = d.groupby(['R_contra', 'P_pn', 'P_ln'])['value'].mean()
    cube = np.stack([g.loc[r].unstack('P_ln').reindex(index=Ppn_ax, columns=Pln_ax).to_numpy()
                     for r in (0.5, 1.0, 2.0)])                     # (3, nPpn, nPln)
    base = float(d[(d['P_pn'] == 0) & (d['P_ln'] == 0)]['value'].mean())
    cube = np.concatenate([np.full((1, len(Ppn_ax), len(Pln_ax)), base), cube], axis=0)
    return RegularGridInterpolator((R_ax, Ppn_ax, tln_ax), cube, bounds_error=False, fill_value=None)

dsi_model = coords[['DSI', 'Odor_type']].copy()
for _name, _addr in MODEL_READOUTS.items():
    dsi_model[f'pred_{_name}'] = model_interpolator(*_addr)(query)
print(f'interpolated {len(MODEL_READOUTS)} model readouts at {len(dsi_model)} glomeruli '
      f'(structural GN1, GRU; R_contra=0 anchored on the unilateral baseline)')
dsi_model.sort_values('DSI', ascending=False)


In [ ]:
# DSI vs model-predicted lateral-coding score — one panel per readout (Pearson + Spearman).
from scipy import stats

plt.style.use('dark_background')
PANEL = [('pred_source_side',     'source-side AUROC',    r'model source-side AUROC'),
         ('pred_log_ratio_conc',  'log-ratio $R^2$',      r'model log-ratio $R^2$'),
         ('pred_contrast_index',  'contrast $R^2$',       r'model contrast $R^2$'),
         ('pred_difference_conc', 'abs-difference $R^2$', r'model abs-difference $R^2$')]
otypes = list(dsi_model['Odor_type'].dropna().unique())
opal = dict(zip(otypes, sns.color_palette('Set2', len(otypes))))

fig, axes = plt.subplots(1, 4, figsize=(20, 5.2))
for ax, (col, short, xlab) in zip(axes, PANEL):
    d = dsi_model[['DSI', col, 'Odor_type']].dropna()
    x, y = d[col].to_numpy(float), d['DSI'].to_numpy(float)
    for ot in otypes:
        s = d[d['Odor_type'] == ot]
        ax.scatter(s[col], s['DSI'], color=opal[ot], s=55, edgecolor='w', linewidth=0.4,
                   label=ot, zorder=2)
    for g, row in d.iterrows():
        ax.annotate(g, (row[col], row['DSI']), fontsize=7, color='0.85',
                    xytext=(3, 3), textcoords='offset points', zorder=3)
    # correlations (two-sided) + least-squares guide line
    pr, pp = stats.pearsonr(x, y)
    sr, sp = stats.spearmanr(x, y)
    slope, intercept = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, slope * xs + intercept, color='white', lw=1, ls='--', alpha=0.7, zorder=1)
    ax.set_xlabel(xlab)
    ax.set_title(f'DSI vs {short}   (n = {len(d)})\n'
                 f'Pearson r = {pr:.2f} (p = {pp:.3g})    Spearman ρ = {sr:.2f} (p = {sp:.3g})',
                 fontsize=9)
axes[0].set_ylabel('DSI  (functional directional selectivity)')
axes[0].legend(title='odour type', fontsize=8, title_fontsize=9, loc='best', framealpha=0.3)
fig.suptitle('Measured DSI vs AL-model–predicted lateral-coding score at real connectome coordinates',
             y=1.03, fontsize=13)
plt.tight_layout()
plt.show()


### 12b. Dataset-specific: DSI vs AL-model predictions

The broad panel above interpolates the pre-registered model grid at the
**female-pool** coordinates. Here the same interpolation is evaluated at **each
animal's own** pooled coordinates (rows = animal, columns = model readout), so
the male/female and dataset-specific behaviour is visible separately.
⚠ The MCNS row inherits the section-6b side-label artifact; FAFB / BANC /
hemibrain are the reliable rows.

In [ ]:
# ── Dataset-specific: DSI vs model-predicted lateral-coding score ─────────────
# Re-uses model_interpolator / MODEL_READOUTS / PANEL / R_ax / Ppn_ax / tln_ax from
# section 12, but queries the grid at EACH animal's own pooled coordinates.
nrow, ncol = len(ANIMALS), len(PANEL)
fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 3.0 * nrow + 1.0), squeeze=False)
for i, a in enumerate(ANIMALS):
    _s = summary[summary['animal'] == a].copy()
    _s['rf'] = _s['ratio'].replace(np.inf, np.nan)
    _conn_a = _s.groupby(['glomerulus', 'metric'])['rf'].median().unstack('metric')[['R_contra', 'P_PN', 'P_LN']]
    _coords = dsi.merge(_conn_a, on='glomerulus', how='left').dropna(subset=['R_contra', 'P_PN', 'P_LN']).set_index('glomerulus')
    qR   = np.clip(_coords['R_contra'].to_numpy(float), R_ax.min(), R_ax.max())
    qPpn = np.clip(_coords['P_PN'].to_numpy(float),       Ppn_ax.min(), Ppn_ax.max())
    qtln = np.clip(np.log1p(_coords['P_LN'].to_numpy(float)), tln_ax.min(), tln_ax.max())
    qq = np.column_stack([qR, qPpn, qtln])
    _dm = _coords[['DSI', 'Odor_type']].copy()
    for _name, _addr in MODEL_READOUTS.items():
        _dm[f'pred_{_name}'] = model_interpolator(*_addr)(qq)
    for j, (col, short, xlab) in enumerate(PANEL):
        ax = axes[i][j]
        dd = _dm[['DSI', col, 'Odor_type']].dropna()
        if len(dd) < 4:
            ax.text(0.5, 0.5, f'no match (n={len(dd)})', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
            continue
        x, y = dd[col].to_numpy(float), dd['DSI'].to_numpy(float)
        for ot in otypes:
            s = dd[dd['Odor_type'] == ot]
            ax.scatter(s[col], s['DSI'], color=opal[ot], s=45, edgecolor='w', linewidth=0.4, zorder=2)
        pr, pp = stats.pearsonr(x, y); sr, sp = stats.spearmanr(x, y)
        slope, intercept = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 50)
        ax.plot(xs, slope * xs + intercept, color='white', lw=1, ls='--', alpha=0.7, zorder=1)
        ax.set_title(f'{a} · {short}\nr = {pr:.2f} (p = {pp:.3g})    ρ = {sr:.2f} (p = {sp:.3g})', fontsize=7.5)
        ax.set_xlabel(xlab if i == nrow - 1 else '')
        if j == 0:
            ax.set_ylabel('DSI')
fig.suptitle("Dataset-specific: measured DSI vs model-predicted lateral-coding score at each animal's own coordinates",
             y=1.0, fontsize=12)
plt.tight_layout()
plt.show()